In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
%%capture
!pip install --no-deps unsloth
!pip install transformers accelerate bitsandbytes peft trl datasets xformers sentencepiece

In [4]:
%%capture
!pip install unsloth_zoo

In [5]:
from unsloth import FastLanguageModel

max_seq_length = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/Qwen2.5-Coder-3B-Instruct",
    max_seq_length = max_seq_length,
    dtype = None,
    load_in_4bit = True,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.5.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 7.5. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors:   0%|          | 0.00/2.05G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/266 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

...

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/632 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

In [35]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    target_modules = [
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
    lora_alpha = 32,
    lora_dropout = 0.05,
    bias = "none",
    use_gradient_checkpointing = "unsloth",
)

TypeError: Unsloth: Your model already has LoRA adapters. Your new parameters are different.

In [13]:
from datasets import load_dataset
from sklearn.model_selection import train_test_split

dataset = load_dataset("nickrosh/Evol-Instruct-Code-80k-v1", split="train")
split_dataset = dataset.train_test_split(test_size=0.1, seed=42)
dataset_train = split_dataset["train"]
dataset_test = split_dataset["test"]

# print("Dataset train: ", dataset_train)
# print("Dataset test: ", dataset_test)
# print(dataset_train[0]['instruction'])
# print(dataset_train[0]['output'])

In [14]:
SYSTEM_PROMPT = "You are a helpful, respectful and honest code assistant. Always answer as helpfully as possible."

def format_qwen_template(examples):
    formatted_messages = []

    # Duyệt qua từng dòng trong lô dữ liệu (batch)
    for instruction, output in zip(examples["instruction"], examples["output"]):
        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": instruction},
            {"role": "assistant", "content": output}
        ]
        formatted_messages.append(messages)

    return {"messages": formatted_messages}

dataset_train = dataset_train.map(
    format_qwen_template,
    batched=True,
    remove_columns=["instruction", "output"],
)

Map:   0%|          | 0/70437 [00:00<?, ? examples/s]

In [18]:
# print(dataset_train[0]['messages'])

In [31]:
def apply_qwen_template(examples):
    texts = [
        tokenizer.apply_chat_template(msg, tokenize=False, add_generation_prompt=False)
        for msg in examples["messages"]
    ]
    return {"text": texts}

dataset_train = dataset_train.map(
    apply_qwen_template,
    batched=True
)

# Apply the same preprocessing to dataset_test
dataset_test = dataset_test.map(
    format_qwen_template,
    batched=True,
    remove_columns=["instruction", "output"],
)
dataset_test = dataset_test.map(
    apply_qwen_template,
    batched=True
)

Map:   0%|          | 0/70437 [00:00<?, ? examples/s]

Map:   0%|          | 0/7827 [00:00<?, ? examples/s]

Map:   0%|          | 0/7827 [00:00<?, ? examples/s]

In [20]:
print(dataset_train[0]['text'])

<|im_start|>system
You are a helpful, respectful and honest code assistant. Always answer as helpfully as possible.<|im_end|>
<|im_start|>user
Given a list of strings, write a program to combine them into one string with a space between each element.
lst = ['This', 'is', 'a', 'list', 'of', 'strings']<|im_end|>
<|im_start|>assistant
result = " ".join(lst)
print(result) # This is a list of strings<|im_end|>



In [37]:
small_eval_dataset = dataset_test.shuffle(seed=42).select(range(200))
print(small_eval_dataset)

Dataset({
    features: ['messages', 'text'],
    num_rows: 200
})


In [38]:
def preprocess_logits_for_metrics(logits, labels):
    if isinstance(logits, tuple):
        logits = logits[0]
    # Lấy vị trí token có xác suất cao nhất (Argmax) thay vì giữ toàn bộ phân phối xác suất
    return logits.argmax(dim=-1)

# 2. Hàm tính toán chỉ số chính xác ở mức độ Token
def compute_metrics(eval_preds):
    preds, labels = eval_preds

    # labels có dạng mã nguồn, các token không cần tính điểm (prompt) sẽ mang giá trị -100
    # Ta tạo một mask để lọc bỏ các vị trí -100 này ra
    mask = labels != -100

    # Lọc ra các token thực sự thuộc phần code phản hồi (output)
    labels_filtered = labels[mask]
    preds_filtered = preds[mask]

    # Tính độ chính xác tương đối (Token-level Accuracy)
    correct = (preds_filtered == labels_filtered).sum()
    total = len(labels_filtered)
    accuracy = correct / total if total > 0 else 0.0

    return {
        "token_accuracy": accuracy
    }

def formatting_func(batch):
    return batch["text"]

In [40]:
from trl import SFTTrainer
from transformers import TrainingArguments, EarlyStoppingCallback
from unsloth.chat_templates import train_on_responses_only
import numpy as np
import torch

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset_train,
    eval_dataset = small_eval_dataset,
    packing = True,
    peft_config = model.peft_config,
    formatting_func = formatting_func,
    max_seq_length = 2048,
    compute_metrics = compute_metrics,     # Đưa hàm tính metric vào
    preprocess_logits_for_metrics = preprocess_logits_for_metrics,

    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 10,
        max_steps = 200,
        learning_rate = 2e-4,
        fp16 = True,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.05,
        lr_scheduler_type = "cosine",
        output_dir = "outputs",

        eval_strategy = "steps",
        eval_steps = 40,
        save_strategy = "steps",
        save_steps = 40,
        save_total_limit = 2,
         per_device_eval_batch_size = 4,

        prediction_loss_only = False,       # Phải đặt False để kích hoạt sinh dự đoán
        load_best_model_at_end = True,
        metric_for_best_model = "token_accuracy",
        greater_is_better = True,
    ),

    callbacks = [
        EarlyStoppingCallback(
            early_stopping_patience = 3,     # Chờ tối đa 3 lần đánh giá không cải thiện
            early_stopping_threshold = 0.0   # Mức độ cải thiện tối thiểu (mặc định là 0.0)
        )
    ],
)

Unsloth: Tokenizing ["text"] (num_proc=5):   0%|          | 0/70437 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=5):   0%|          | 0/200 [00:00<?, ? examples/s]

In [41]:
trainer = train_on_responses_only(
    trainer,
    instruction_part = "<|im_start|>user",
    response_part = "<|im_start|>assistant",
)

Map (num_proc=5):   0%|          | 0/70437 [00:00<?, ? examples/s]

Filter (num_proc=5):   0%|          | 0/70437 [00:00<?, ? examples/s]

Unsloth: Removed 10 out of 70437 samples from train_dataset where all labels were -100 (no response found after truncation). This prevents NaN loss during training.


Map (num_proc=5):   0%|          | 0/200 [00:00<?, ? examples/s]

Filter (num_proc=5):   0%|          | 0/200 [00:00<?, ? examples/s]

In [42]:
trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 70,427 | Num Epochs = 1 | Total steps = 200
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 29,933,568 of 3,115,872,256 (0.96% trained)


Step,Training Loss,Validation Loss,Token Accuracy
40,0.146743,0.342577,0.003615
80,0.321681,0.328053,0.003674
120,0.268066,0.322734,0.003694
160,0.259821,0.321322,0.003694
200,0.178651,0.320964,0.003694


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

TrainOutput(global_step=200, training_loss=0.29647743195295334, metrics={'train_runtime': 997.5348, 'train_samples_per_second': 1.604, 'train_steps_per_second': 0.2, 'total_flos': 1.399234829451264e+16, 'train_loss': 0.29647743195295334, 'epoch': 0.022718237064803772})

In [43]:
model.save_pretrained_merged("final_lora_model", tokenizer, save_method = "lora")

config.json:   0%|          | 0.00/764 [00:00<?, ?B/s]

Unsloth: Restored added_tokens_decoder metadata in final_lora_model/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...
Cache check failed: model-00001-of-00002.safetensors not found in local cache.
Not all required files found in cache. Will proceed with downloading.
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...

model-00002-of-00002.safetensors:   0%|          | 0.00/1.21G [00:00<?, ?B/s]


Unsloth: Preparing safetensor model files: 100%|██████████| 2/2 [12:55<00:00, 387.52s/it]


Note: tokenizer.model not found (this is OK for non-SentencePiece models)



Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [03:10<00:00, 95.11s/it]


Unsloth: Merge process complete. Saved to `/content/final_lora_model`


In [ ]:
import random
from unsloth import FastLanguageModel
import torch

max_seq_length = 2048
dtype = None
load_in_4bit = True

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "final_lora_model",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

FastLanguageModel.for_inference(model)

sample_indices = random.sample(range(len(dataset_test)), 3)

print("============ BẮT ĐẦU KIỂM THỬ CHẤT LƯỢNG SINH CODE ============\n")

for idx in sample_indices:
    sample_question = dataset_test[idx]["instruction"] # Lấy câu hỏi gốc
    gold_answer = dataset_test[idx]["output"]         # Lấy đáp án code mẫu để đối chiếu

    # Định dạng câu hỏi theo đúng chuẩn cấu hình Chat Template của Qwen2.5
    messages = [
        {"role": "user", "content": sample_question}
    ]
    inputs = tokenizer.apply_chat_template(
        messages,
        tokenize = True,
        add_generation_prompt = True,
        return_tensors = "pt",
    ).to("cuda")

    with torch.no_grad():
        outputs = model.generate(
            input_ids = inputs,
            max_new_tokens = 512,        # Giới hạn độ dài đoạn code sinh ra tối đa 512 tokens
            use_cache = True,            # Bật cache để tăng tốc độ sinh từ tiếp theo
            temperature = 0.2,           # Đặt thấp (0.1 - 0.2) để code sinh ra có tính logic, chuẩn xác cao
            top_p = 0.9,
        )


    generated_text = tokenizer.decode(outputs[0][inputs.shape[1]:], skip_special_tokens=True)

    # 5. In kết quả trực quan ra màn hình để đối chiếu mắt thường
    print(f"--- [Mẫu dữ liệu số {idx}] ---")
    print(f"YÊU CẦU BÀI TOÁN (Prompt):\n{sample_question}\n")
    print(f"CODE MÔ HÌNH TỰ SINH (Generated Code):\n{generated_text}\n")
    print(f"CODE MẪU ĐỐI CHIẾU (Dataset Target):\n{gold_answer}\n")
    print("-" * 60 + "\n")